<img src="http://www.cidaen.es/assets/img/mCIDaeNnb.png" alt="Logo CiDAEN" align="right">


<br><br><br>
<h2><font color="#00586D" size=4>Módulo 12: Arquitecturas y procesos Big Data</font></h2>



<h1><font color="#00586D" size=5>Capstone 12. Parte 1: Modelo de <i>sentiment</i> sobre Amazon Reviews</font></h1>

<br><br><br>
<div style="text-align: right">
<font color="#00586D" size=3>Enrique González, Jacinto Arias</font><br>
<font color="#00586D" size=3>Máster en Ciencia de Datos e Ingeniería de Datos en la Nube</font><br>
<font color="#00586D" size=3>Universidad de Castilla-La Mancha</font>




</div>

<a id="indice"></a>
<h2><font color="#00586D" size=5>Índice</font></h2>


* [1. Introducción](#section1)
* [2. Análisis exploratorio](#section2)
* [3. Modelado](#section3)

A continuación vamos a preparar el entorno. Recordad que tenéis que ingresar vuestras credenciales de AWS Academy en el fichero .env antes de levantar Jupyter Lab.

In [5]:
%iam_role arn:aws:iam::270830193283:role/LabRole 
%region us-east-1
%number_of_workers 2
%idle_timeout 30

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
It looks like there is a newer version of the kernel available. The latest version is 1.0.9 and you have 1.0.4 installed.
Please run `pip install --upgrade aws-glue-sessions` to upgrade your kernel
Current iam_role is None
iam_role has been set to arn:aws:iam::270830193283:role/LabRole.
Previous region: None
Setting new region to: us-east-1
Region is set to: us-east-1
Previous number of workers: None
Setting new number of workers to: 2
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.


In [1]:
s3_bucket = "s3://es.cidaen.m12.dataframes.luis"

Trying to create a Glue session for the kernel.
Session Type: etl
Worker Type: G.1X
Number of Workers: 2
Session ID: 96c92309-2658-4960-85e5-17e20d7fdd79
Applying the following default arguments:
--glue_kernel_version 1.0.4
--enable-glue-datacatalog true
Waiting for session 96c92309-2658-4960-85e5-17e20d7fdd79 to get into ready status...
Session 96c92309-2658-4960-85e5-17e20d7fdd79 has been created.



In [11]:
!ls

capstone_test.ipynb  capstone_training.ipynb  data  img


A continuación podéis subir los datos con los que trabajaremos en esta práctica. Estos datos los podéis encontrar en el Campus Virtual del módulo bajo el nombre `amazon-reviews-pds-parquet`. 

In [7]:
!aws s3 cp --recursive ./data/amazon-reviews-pds-parquet s3://es.cidaen.m12.dataframes.luis.2/amazon-reviews-pds-parquet

upload: data/amazon-reviews-pds-parquet/_SUCCESS to s3://es.cidaen.m12.dataframes.luis.2/amazon-reviews-pds-parquet/_SUCCESS
upload: data/amazon-reviews-pds-parquet/._SUCCESS.crc to s3://es.cidaen.m12.dataframes.luis.2/amazon-reviews-pds-parquet/._SUCCESS.crc
upload: data/amazon-reviews-pds-parquet/product_category=Automotive/.part-00001-630e930d-d532-4008-bb87-a2d5dd9fdae9.c000.snappy.parquet.crc to s3://es.cidaen.m12.dataframes.luis.2/amazon-reviews-pds-parquet/product_category=Automotive/.part-00001-630e930d-d532-4008-bb87-a2d5dd9fdae9.c000.snappy.parquet.crc
upload: data/amazon-reviews-pds-parquet/product_category=Automotive/.part-00000-630e930d-d532-4008-bb87-a2d5dd9fdae9.c000.snappy.parquet.crc to s3://es.cidaen.m12.dataframes.luis.2/amazon-reviews-pds-parquet/product_category=Automotive/.part-00000-630e930d-d532-4008-bb87-a2d5dd9fdae9.c000.snappy.parquet.crc
upload: data/amazon-reviews-pds-parquet/product_category=Automotive/.part-00010-630e930d-d532-4008-bb87-a2d5dd9fdae9.c000.

Asimismo como en este capstone necesitaremos guardar modelos en s3, necesitaremos añadir la siguiente configuración:

In [2]:
sc._jsc.hadoopConfiguration().set("mapred.output.committer.class", "org.apache.hadoop.mapred.DirectFileOutputCommitter")

---

<a id="section1"></a>
## <font color="#00586D"> 1. Introducción</font>
<br>

En este capstone vamos a aprender un modelo de detección del sentimiento utilizando MLlib y Glue. 

Para ello utilizaremos el dataset de __amazon reviews__. Este dataset tiene las siguientes columnas (de su diccionario de datos):
```
marketplace       - 2 letter country code of the marketplace where the review was written.
customer_id       - Random identifier that can be used to aggregate reviews written by a single author.
review_id         - The unique ID of the review.
product_id        - The unique Product ID the review pertains to. In the multilingual dataset the reviews
                    for the same product in different countries can be grouped by the same product_id.
product_parent    - Random identifier that can be used to aggregate reviews for the same product.
product_title     - Title of the product.
product_category  - Broad product category that can be used to group reviews 
                    (also used to group the dataset into coherent parts).
star_rating       - The 1-5 star rating of the review.
helpful_votes     - Number of helpful votes.
total_votes       - Number of total votes the review received.
vine              - Review was written as part of the Vine program.
verified_purchase - The review is on a verified purchase.
review_headline   - The title of the review.
review_body       - The review text.
review_date       - The date the review was written.
```

De estas, la columna `product_category` se usa como clave de partición. Podéis encontrar toda la información en el enlace que os proporcionamos más arriba. 

---

<a id="section2"></a>
## <font color="#00586D"> 2. Análisis exploratorio</font>
<br>

Antes de empezar con el modelado exploraremos los datos minimamente para poder estudiar sus propiedades.

---
### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 1: Carga de datos </font>
<br>

Carga el dataset completo en formato parquet y cuenta sus registros. De momento, no lo persistas. 




In [10]:
df = spark.read.parquet("s3://es.cidaen.m12.dataframes.luis.2/amazon-reviews-pds-parquet/")
df.count()


16967903


**Resultado esperado**: 16967903 registros

<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>


---
### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 2: Filtrado </font>
<br>


Como el dataset es masivo para entrenar el modelo de sentiment vamos a trabajar únicamente con una partición. Concretamente utilizaremos la partición de `Electronics`. Filtra los datos para quedarte con esta partición y cuenta ahora el total de elementos de este nuevo dataset. No cachees este dataset. 



In [11]:
df_electronics = spark.read.parquet("s3://es.cidaen.m12.dataframes.luis.2/amazon-reviews-pds-parquet/product_category=Electronics/")
df_electronics.count()



3105119


**Resultado esperado**: 3105119 registros

<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>


---
### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 3: Almacenamiento </font>
<br>


Para no seguir trabajando con los datos públicos, vamos a escribir los datos en un bucket de S3 en nuestra cuenta. Para ello, crea un bucket the S3 para este capstone y escribe los datos dentro del bucket en el directorio `electronics`. Utiliza repartition para tener 32 particiones. Tras esto, vuelve a cargar el dataset y cachéalo. 



In [12]:
output_path = "s3://cidaen-capstone-12-1/electronics/"

df_electronics.repartition(32).write.mode("overwrite").parquet(output_path)


In [14]:
df_electronics_new = spark.read.parquet(output_path)

df_electronics_new.cache()

df_electronics_new.count()




3105119




<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>


---
### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 4: Consultas </font>
<br>

Obten los siguiente resultados del dataset que acabáis de cargar:

1. Muestra el total de reviews para cada posible número de estrellas recibidas (*star_rating*)
2. Obtén los 10 productos con mayor número de votos (*total_votes*) mostrando su nombre, numero de votos y valoración media (*star_rating*)
3. Obtén la cantidad de reviews (1 registro de dataset -> 1 review) y la valoración media (*star_rating*) por mes y año. Obten los últimos 15 registros ordenador por año y mes.  

In [41]:
from pyspark.sql.functions import col

df_electronics_new.groupBy("star_rating").count().orderBy(col("star_rating")).show()


+-----------+-------+
|star_rating|  count|
+-----------+-------+
|          1| 359248|
|          2| 179834|
|          3| 239459|
|          4| 538824|
|          5|1787754|
+-----------+-------+


In [44]:
from pyspark.sql.functions import col, avg, sum as spark_sum

df_top10 = (
    df_electronics_new
    .groupBy("product_id", "product_title")
    .agg(
        spark_sum("total_votes").alias("total_votes"),
        avg("star_rating").alias("avg_star_rating")
    )
    .orderBy(col("total_votes").desc())
    .limit(10)
)

df_top10.show(truncate=False)


+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------------+
|product_id|product_title                                                                                                                                                                                       |total_votes|avg_star_rating   |
+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------------+
|B000I1X6PM|Denon AKDL1 Dedicated Link Cable (Discontinued by Manufacturer)                                                                                                                                     |46348      |3.4917491749174916|
|B000J36XR2|AudioQuest K2 Terminated

In [45]:
df_top10 = (
    df_electronics_new
    .groupBy("product_title")
    .agg(
        spark_sum("total_votes").alias("total_votes"),
        avg("star_rating").alias("avg_star_rating")
    )
    .orderBy(col("total_votes").desc())
    .limit(10)
)

df_top10.show(truncate=False)


+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------------+
|product_title                                                                                                                                                                                       |total_votes|avg_star_rating   |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------------+
|Denon AKDL1 Dedicated Link Cable (Discontinued by Manufacturer)                                                                                                                                     |46348      |3.4917491749174916|
|Panasonic ErgoFit In-Ear Earbud Headphone                                      

No coincide el numero de total votes pero el numero de registros si, en el resultado esperado el mismo producto aparece varias veces... por que?

In [47]:
from pyspark.sql.functions import year, month, col, avg, count


df_reviews_by_month = (
    df_electronics_new
    .withColumn("year", year(col("review_date")))
    .withColumn("month", month(col("review_date")))
    .groupBy("year", "month")
    .agg(
        count("*").alias("num_reviews"),
        avg("star_rating").alias("avg_star_rating")
    )
    .orderBy(col("year").desc(), col("month").desc())
    .limit(15)
)

df_reviews_by_month.show(truncate=False)


+----+-----+-----------+------------------+
|year|month|num_reviews|avg_star_rating   |
+----+-----+-----------+------------------+
|2015|8    |102984     |4.093985473471608 |
|2015|7    |99806      |4.08580646454121  |
|2015|6    |91486      |4.093478783639027 |
|2015|5    |89357      |4.100439808856609 |
|2015|4    |93152      |4.102466935760907 |
|2015|3    |108861     |4.11561532596614  |
|2015|2    |107291     |4.118062092813004 |
|2015|1    |120404     |4.152602903558021 |
|2014|12   |107891     |4.120232456831432 |
|2014|11   |77529      |4.10810148460576  |
|2014|10   |78128      |4.114210014335449 |
|2014|9    |77753      |4.116111275449179 |
|2014|8    |82143      |4.115664146671049 |
|2014|7    |79424      |4.118352135374698 |
|2014|6    |48375      |4.0157726098191215|
+----+-----+-----------+------------------+


**Resultados esperados**:
1. Muestra el total de reviews para cada posible número de estrellas recibidas (*star_rating*)

| star_rating | count   |
|-------------|---------|
| 1           | 359248  |
| 5           | 1787754 |
| 2           | 179834  |
| 3           | 239459  |
| 4           | 538824  |


2. Obtén los 10 productos con mayor número de votos (*total_votes*) mostrando su nombre, numero de votos y valoración media (*star_rating*)

| product_title                                                                                  | total_votes | star_rating |
|------------------------------------------------------------------------------------------------|-------------|-------------|
| Denon AKDL1 Dedicated Link Cable (Discontinued by Manufacturer)                                 | 12944       | 3           |
| AudioQuest K2 Terminated Speaker Cable - UST 2.44 m Plugs 8' Pair (Discontinued by Manufacturer) | 9072        | 1           |
| Panasonic ErgoFit In-Ear Earbud Headphone                                                       | 8680        | 5           |
| Apple iPod touch 8GB (4th Generation)                                                          | 6353        | 5           |
| Denon AKDL1 Dedicated Link Cable (Discontinued by Manufacturer)                                 | 5546        | 1           |
| Apple iPod touch 8 GB 2nd Generation                                                           | 4595        | 5           |
| Bose QuietComfort 15 Acoustic Noise Cancelling Headphones (Discontinued by Manufacturer)        | 4556        | 4           |
| Panasonic ErgoFit In-Ear Earbud Headphone                                                       | 4341        | 5           |
| X-Mini II XAM4-B Portable Capsule Speaker, Mono                                                | 4260        | 1           |
| Denon AKDL1 Dedicated Link Cable (Discontinued by Manufacturer)                                 | 4242        | 2           |


3. Obtén la cantidad de reviews (1 registro de dataset -> 1 review) y la valoración media (*star_rating*) por mes y año. Obten los últimos 15 registros ordenador por año y mes. 

| year | month | review_count | mean_star_rating |
|------|-------|--------------|------------------|
| 2015 | 8     | 102984       | 4.094            |
| 2015 | 7     | 99806        | 4.086            |
| 2015 | 6     | 91486        | 4.093            |
| 2015 | 5     | 89357        | 4.100            |
| 2015 | 4     | 93152        | 4.102            |
| 2015 | 3     | 108861       | 4.116            |
| 2015 | 2     | 107291       | 4.118            |
| 2015 | 1     | 120404       | 4.153            |
| 2014 | 12    | 107891       | 4.120            |
| 2014 | 11    | 77529        | 4.108            |
| 2014 | 10    | 78128        | 4.114            |
| 2014 | 9     | 77753        | 4.116            |
| 2014 | 8     | 82143        | 4.116            |
| 2014 | 7     | 79424        | 4.118            |
| 2014 | 6     | 48375        | 4.016            |



<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>


---

<a id="section3"></a>
## <font color="#00586D"> 3. Modelado</font>
<br>

Como paso previo al modelado realizaremos dos procesos de limpieza sobre los datos:


---
### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 5: Preparación del texto </font>
<br>

Limpiead el texto de las reviews (`review_body`) utilizando expresiones sobre strings o expresiones regulares
 - Pasar todo el texto a minusculas.
 - Eliminar números y signos de puntuacion.
 - Si existen, elimina los registros con valores nulos en el body con las transformaciones anteriores. 
 
Muestra los resultados para las primeras 10 filas del dataframe ordenadas por `review_id`

In [15]:
from pyspark.sql.functions import col, lower, regexp_replace, asc
df_limpio = (
    df_electronics
    .withColumn("review_body_clean", lower(col("review_body")))
    .withColumn("review_body_clean", regexp_replace(col("review_body_clean"), r"[^a-zA-Z\s]", ""))
    .filter(col("review_body_clean").isNotNull())
)


In [19]:
(
    df_limpio
    .orderBy(asc("review_id"))
    .select("review_id", "review_body", "review_body_clean")
    .limit(10)
    .toPandas()
)


        review_id  ...                                  review_body_clean
0  R10000WMGXS51T  ...  great little emergency radio  very good recept...
1  R10001L4QTCA84  ...  lives up to its claim and really does fit bulk...
2  R10003OLR2P5UE  ...  ive gone through three pairs of these in the l...
3  R10005O193PJ6W  ...  stopped working after a while changed batterie...
4  R10008LR7CU84N  ...  i ordered this cable and it doesnt work when i...
5  R10009JN2UWOJC  ...  have not owned it that long however it has the...
6  R1000AMVKPW32O  ...  bought for a gift and it is just what was need...
7  R1000CJMO2L8X4  ...                                perfect for the gym
8  R1000EDGJUU3CU  ...  love these  the sound quality is amazing  the ...
9  R1000EG9XXBLXT  ...  i have had good success with these disks and h...

[10 rows x 3 columns]


**Resultado esperado**:

| review_id     | review_body                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      | clean_review_body                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
|---------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| R10000WMGXS51T | Great little emergency radio.  Very good reception.  The<br />weather band is a feature.  Can't beat the quality for this price/                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    | great little emergency radio  very good reception  thebr weather band is a feature  cant beat the quality for this price                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               |
| R10001L4QTCA84| Lives up to its claim, and really does fit bulky phone cases. Braided cable is sturdy but flexible. I think it stays a little more flexible in the cold weather, which is nice. Definitely getting a few more in the future!                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   | lives up to its claim and really does fit bulky phone cases braided cable is sturdy but flexible i think it stays a little more flexible in the cold weather which is nice definitely getting a few more in the future                                                                                                                                                                                                                                                                                                                                                                                                                                                |
| R10003OLR2P5UE| I've gone through three pairs of these in the last two years. I am in love with the sound quality, and even though I know it's not the best I particularly love how the bass sounds. They're comfortable to wear and very isolating. With these headphones, you don't even need noise canceling. There is very little sound leak, unless you like to listen to music ridiculously loud. All in all, I was very impressed with these. They're without a doubt the best sounding headphones I've ever owned.<br /><br />Now, the problem: The wires are thin and stringy, and do NOT last. On my first pair, the part of the wire that connected to the left cup came apart. I'm not an abuser of headphones, either. On the other two pairs, they wire at the base next to the adapter came apart. I went at them with a soldering iron, desperately trying to make them last as long as I could, but they'd always crap out on me again. The sound quality distorts over time, and the foam around the cups is cheap and wears out quickly.<br /><br />They aren't worth the price for such bad quality. I'd suggest looking around for other pairs, Sony, Denon, and Sennheiser all have superior headphones for a similar price. I myself just ordered a pair of Denon AHD1001's, and here's hoping they last longer! | ive gone through three pairs of these in the last two years i am in love with the sound quality and even though i know its not the best i particularly love how the bass sounds theyre comfortable to wear and very isolating with these headphones you dont even need noise canceling there is very little sound leak unless you like to listen to music ridiculously loud all in all i was very impressed with these theyre without a doubt the best sounding headphones ive ever ownedbr br now the problem the wires are thin and stringy and do not last on my first pair the part of the wire that connected to the left cup came apart im not an abuser of headphones either on the other two pairs they wire at the base next to the adapter came apart i went at them with a soldering iron desperately trying to make them last as long as i could but theyd always crap out on me again the sound quality distorts over time and the foam around the cups is cheap and wears out quicklybr br they arent worth the price for such bad quality id suggest looking around for other pairs sony denon and sennheiser all have superior headphones for a similar price i myself just ordered a pair of denon ahds and heres hoping they last longer |
| R10005O193PJ6W| stopped working after a while, changed batteries, it worked for a few days, then it quit                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             | stopped working after a while changed batteries it worked for a few days then it quit                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      |
| R10008LR7CU84N| I ordered this cable and it doesn't work when I contacted them they told me I was doing something wrong. I then had my dad who it a certified computer tech look at it and there is something wrong with the cable. When I told them they never responded to me again.                                                                                                                                                                                                                                                                                                                                                                                                                                                     | i ordered this cable and it doesnt work when i contacted them they told me i was doing something wrong i then had my dad who it a certified computer tech look at it and there is something wrong with the cable when i told them they never responded to me again                                                                                                                                                                                                                                                                                                                                                                                               |
| R10009JN2UWOJC| Have not owned it that long however it has the features , feel and works like a quality unit that would be at a much higher price point                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 | have not owned it that long however it has the features  feel and works like a quality unit that would be at a much higher price point                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    |
| R1000AMVKPW32O| Bought for a gift and it is just what was needed to mount the new 32" TV outdoors. The fact that it has full motion swing makes it even better because we can move it around to see it from different angles and still have a sturdy mount.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            | bought for a gift and it is just what was needed to mount the new  tv outdoors the fact that it has full motion swing makes it even better because we can move it around to see it from different angles and still have a sturdy mount                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| R1000CJMO2L8X4| Perfect for the gym                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 | perfect for the gym                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| R1000EDGJUU3CU| Love these !!! The sound quality is amazing ! The price was amazing especially for the quality.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      | love these  the sound quality is amazing  the price was amazing especially for the quality                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |
| R1000EG9XXBLXT| I have had good success with these disks, and have used hundreds of them successfully on both computers and a dedicated Panosonic DVD recorder. They seem very reliable, and the lines on the disk label help to keep labeling neat and straight.                                                                                                                                                                                                                                                                                                                                                                                                                                                                              | i have had good success with these disks and have used hundreds of them successfully on both computers and a dedicated panosonic dvd recorder they seem very reliable and the lines on the disk label help to keep labeling neat and straight                                                                                                                                                                                                                                                                                                                                                                                                                        |


<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>


---
### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 6: Obtención del sentiment </font>
<br>

Cread la variable `sentiment` en función del número de estrellas asumiendo que una review de menos (<) de 3 estrellas es negativa, usando 1 para el sentiment positivo y 0 para el negativo. Para poder generar la variable que determine el sentiment a partir del número de estrellas podéis utilizar la función de spark `when`. Muestra el resultado para las primeras 10 reviews ordenadas por `review_id`. 

In [49]:
from pyspark.sql.functions import when, col, asc

df_sentiment = (
    df_limpio
    .withColumn(
        "sentiment",
        when(col("star_rating") < 3, 0).otherwise(1)
    )
    .orderBy(asc("review_id"))
)

df_sentiment_pd = (
    df_sentiment
    .select("review_id", "review_body", "star_rating", "sentiment")
    .limit(10)
    .toPandas()
)

df_sentiment_pd



        review_id  ...                                  review_body_clean
0  R10000WMGXS51T  ...  great little emergency radio  very good recept...
1  R10001L4QTCA84  ...  lives up to its claim and really does fit bulk...
2  R10003OLR2P5UE  ...  ive gone through three pairs of these in the l...
3  R10005O193PJ6W  ...  stopped working after a while changed batterie...
4  R10008LR7CU84N  ...  i ordered this cable and it doesnt work when i...
5  R10009JN2UWOJC  ...  have not owned it that long however it has the...
6  R1000AMVKPW32O  ...  bought for a gift and it is just what was need...
7  R1000CJMO2L8X4  ...                                perfect for the gym
8  R1000EDGJUU3CU  ...  love these  the sound quality is amazing  the ...
9  R1000EG9XXBLXT  ...  i have had good success with these disks and h...

[10 rows x 4 columns]


In [54]:
df_sentiment.select("review_id", "review_body", "star_rating", "sentiment").show(10)

+--------------+--------------------+-----------+---------+
|     review_id|         review_body|star_rating|sentiment|
+--------------+--------------------+-----------+---------+
|R10000WMGXS51T|Great little emer...|          5|        1|
|R10001L4QTCA84|Lives up to its c...|          5|        1|
|R10003OLR2P5UE|I've gone through...|          3|        1|
|R10005O193PJ6W|stopped working a...|          3|        1|
|R10008LR7CU84N|I ordered this ca...|          1|        0|
|R10009JN2UWOJC|Have not owned it...|          5|        1|
|R1000AMVKPW32O|Bought for a gift...|          5|        1|
|R1000CJMO2L8X4| Perfect for the gym|          5|        1|
|R1000EDGJUU3CU|Love these !!! Th...|          5|        1|
|R1000EG9XXBLXT|I have had good s...|          5|        1|
+--------------+--------------------+-----------+---------+
only showing top 10 rows


**Resultado esperado**:

| review_id      | review_body                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             |   star_rating |   sentiment |
|:---------------|:------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|--------------:|------------:|
| R10000WMGXS51T | Great little emergency radio.  Very good reception.  The<br />weather band is a feature.  Can't beat the quality for this price/                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        |             5 |           1 |
| R10001L4QTCA84 | Lives up to its claim, and really does fit bulky phone cases. Braided cable is sturdy but flexible. I think it stays a little more flexible in the cold weather, which is nice. Definitely getting a few more in the future!                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            |             5 |           1 |
| R10003OLR2P5UE | I've gone through three pairs of these in the last two years. I am in love with the sound quality, and even though I know it's not the best I particularly love how the bass sounds. They're comfortable to wear and very isolating. With these headphones, you don't even need noise canceling. There is very little sound leak, unless you like to listen to music ridiculously loud. All in all, I was very impressed with these. They're without a doubt the best sounding headphones I've ever owned.<br /><br />Now, the problem: The wires are thin and stringy, and do NOT last. On my first pair, the part of the wire that connected to the left cup came apart. I'm not an abuser of headphones, either. On the other two pairs, they wire at the base next to the adapter came apart. I went at them with a soldering iron, desperately trying to make them last as long as I could, but they'd always crap out on me again. The sound quality distorts over time, and the foam around the cups is cheap and wears out quickly.<br /><br />They aren't worth the price for such bad quality. I'd suggest looking around for other pairs, Sony, Denon, and Sennheiser all have superior headphones for a similar price. I myself just ordered a pair of Denon AHD1001's, and here's hoping they last longer! |             3 |           1 |
| R10005O193PJ6W | stopped working after a while, changed batteries, it worked for a few days, then it quit                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                |             3 |           1 |
| R10008LR7CU84N | I ordered this cable and it doesn't work when I contacted them they told me I was doing something wrong. I then had my dad who it a certified computer tech look at it and there is something wrong with the cable. When I told them they never responded to me again.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  |             1 |           0 |
| R10009JN2UWOJC | Have not owned it that long however it has the features , feel and works like a quality unit that would be at a much higher price point                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |             5 |           1 |
| R1000AMVKPW32O | Bought for a gift and it is just what was needed to mount the new 32&#34; TV outdoors. The fact that it has full motion swing makes it even better because we can move it around to see it from different angles and still have a sturdy mount.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |             5 |           1 |
| R1000CJMO2L8X4 | Perfect for the gym                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     |             5 |           1 |
| R1000EDGJUU3CU | Love these !!! The sound quality is amazing ! The price was amazing especially for the quality.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |             5 |           1 |
| R1000EG9XXBLXT | I have had good success with these disks, and have used hundreds of them successfully on both computers and a dedicated Panosonic DVD recorder. They seem very reliable, and the lines on the disk label help to keep labeling neat and straight.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |             5 |           1 |







<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>


---
### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 7: División del conjunto de datos </font>
<br>

Divide el conjunto de datos en entrenamiento (70% de los datos) y test (30% de los datos). Una vez hecho esto, guarda los datos de test en el bucket the s3 previamente creado (usa el nombre `electronics_test`)

In [55]:
train_df, test_df = df_sentiment.randomSplit([0.7, 0.3], seed=42)


In [56]:
test_df.write.mode("overwrite").parquet("s3://cidaen-capstone-12-1/electronics_test/")





<div style="text-align: right"><font size=4> <i class="fa fa-check-square-o" aria-hidden="true" style="color:#00586D"></i></font></div>

-----

A continuación vamos a entrenar el modelo, para ello utilizaremos diferentes opciones de preprocesamiento. Para poder entrenar un clasificador de sentimiento necesitamos contruir una representación del texto que nos permita entrenar el modelo. Para ello utilizaremos podemos utilizar algoritmos de extracción de características como TF-IDF o Word2Vec que vienen implementados en Spark MLlib y que nos permitirá transformar una cadena de texto a un vector para utilizarlo como datos de entrenamiento de un clasificador.  

---
### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 8: Modelo de sentiment </font>
<br>

Construye un **pipeline** de entrenamiento de un modelo de sentiment a partir de los datos preparados anteriormente. Deberas utilizar una secuencia de diferentes  **transformadores** y **estimadores**:
- `Tokenizer` nos permitirá construir un vector de palabras a partir de nuestras sentencias
- `StopWordsRemover` nos permitirá limpiar de nuestros vectores de palabras las de menor significado

- Construcción de características dos alternativas:
    -  Modelo TF-IDF usando `HashingTF` e `IDF`
    - `Word2Vec` nos permitirá crear un vector a partir de la lista de palabras

- Clasificación binaria, basada en la variable sentiment que hemos utilizado, aplica un clasificador (LogisticRegregession, DecisionTree) evita ensembles por su alto tiempo de aprendizaje.

Buscando en la documentación, encuentra los distintos elementos y conectalos en un pipeline junto a un algorimo de clasificación

- Recomendamos utilizar una muestra (método `sample`) pues el tiempo puede ser excesivo
- Es posible ajustar hiperparámetros, pero igualmente puede ser bastante lento

**Valida el modelo con el conjunto de test anterior usando el area bajo la curva ROC**

In [57]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Tokenizer, StopWordsRemover, Word2Vec
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator


In [58]:
train_sample = train_df.sample(withReplacement=False, fraction=0.05, seed=42)
test_sample = test_df.sample(withReplacement=False, fraction=0.05, seed=42)



In [59]:
tokenizer = Tokenizer(inputCol="review_body_clean", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
word2vec = Word2Vec(inputCol="filtered_words", outputCol="features", vectorSize=100, minCount=5)
lr = LogisticRegression(featuresCol="features", labelCol="sentiment")

pipeline = Pipeline(stages=[tokenizer, remover, word2vec, lr])


In [60]:
model = pipeline.fit(train_sample)

---
### <font color="#004D7F"> <i class="fa fa-pencil-square-o" aria-hidden="true" style="color:#004D7F"></i> Tarea 9: Serialización </font>
<br>

Guarda el modelo entrenado en S3 (al bucket que creaste anteriormente) utilizando la opción nativa de Spark. 

In [62]:
model.save("s3://cidaen-capstone-12-1/word2vec_model/")
